# BUSINESS UNDERTANDING
#### >Churn adalah metrik bisnis yang mengukur jumlah pelanggan yang berhenti menggunakan produk atau layanan

##### >> Telco adalah perusahaan telekomunikasi yang menyediakan layanan seperti internet, telepon, dan layanan tambahan lainnya. Dalam project ini, dataset digunakan untuk menganalisis faktor-faktor yang berkaitan dengan customer churn pada perusahaan telco.

Tujuan project ini adalah menganaisis faktor yang mempengaruhi customer churn pada perusahaan telco dan membuat model sederhana untuk mengidentifikasi pelanggan berisiko churn

## Pertanyaan bisnisnya

1. Berapa churn rate customer?
2. Customer seperti apa yang seting churn
3. Faktor apa yang paling berhubungan dengan churn
4. segment mana yang harus diprioritaskan untuk retention



In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Load Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head(10)

# EDA (Exploratory Data Analysis)

In [ ]:
# ukuran data(baris, kolom)
df.shape

In [ ]:
#ini lebih jelas (baris, kolom) dan tipe data
df.info()

## DATA UNDERSTANDING

#### 1. apakah ada missing value?
#### 2. apakah ada duplikasi?
#### 3. jumlah kategori di tiap kolom
#### 4. distibusi target churn

In [ ]:
# ringkasan deskriptif numerik
df.describe()

##### >>> tenure adalah lama masa berlangganan
##### >>> rata rata 0.162147 di senior citizen(orang tua) artinya adalah 16.2 % adalah orang tua, 83.8% bukan orang tua





In [ ]:
df.describe(include='object')

In [ ]:
# Mencari nilai kosong dan duplikat

print("Missing Values:")
print(df.isna().sum())


In [ ]:
# Mencari nilai kosong dan duplikat

print("Duplicate Values:")
print(df.duplicated().sum())

In [ ]:
df['Churn'].value_counts()


In [ ]:
df['Churn'].value_counts(normalize=True) * 100

# Target Variable: Churn 


> berdasarkan data yang ada 73.5% tidak berhenti berlangganan

> sedangkan 26.5% berhenti berlangganan atau churn. ini yang menjadi perhatian selanjutnya

In [ ]:
df.nunique()

> ada yang aneh pada salah satu kolom di dataset ini, yaitu "**TotalCharges**" karena dia bentuknya object dan bukan angka padahal harusnya isinya angka karena itu adalah nilai dari total bayar

In [ ]:
df['TotalCharges'].dtype

In [ ]:
df['TotalCharges'].head(10)

> Disini dari **"TotalCharges"** datanya angka bukan txt atau char
> jadi harus di ubah menjadi angka

### Mengubah tipe data dari TotalCharges

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

> errors='coerse' adalah kalau ada nilai yang gagal diubah jadi angka, ubah menjadi NaN / missing value.



In [ ]:
df.info()

In [ ]:
df['TotalCharges'].describe()

In [ ]:
df.describe()

#### Melihat data kosong di TotalCharges

In [ ]:
print("Missing Value TotalCharges")
print(df['TotalCharges'].isna().sum())

#### Menhapus atau drop data dari TotalCharges karena cuma sedikit

In [ ]:
df = df.dropna(subset=['TotalCharges'])
df.info()

> **subset** artinya mengambil kolom tertentu

> formatnya itu subset=['kolom1', 'kolom2']

In [ ]:
df.isna().sum()

## Setelah melakukan EDA tahap pertama

#### 1. berapa banyak customer yang churn?

In [ ]:
print("Data per customer:")
print(df['Churn'].value_counts())

print("\nPersentase Data:")
print(df['Churn'].value_counts(normalize= True) * 100 )

## Visualiasi Customer Churn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=df, x='Churn')
plt.title('Customer Churn Distribution')
plt.show()

## Mencari kolom mana yang sangat berkaitan dengan orang tidak mau berlangganan lagi atau churn

> hue adalah pembeda warnanya, pada kasus ini menjadi pembeda dari kolom dependents

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

cat_cols.remove('customerID')
cat_cols.remove('Churn')

n = len(cat_cols)
fig, axes = plt.subplots(nrows=(n//3)+1, ncols=3, figsize=(18, n*2))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, hue='Churn', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
    
plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
num_cols.remove('SeniorCitizen')

fig, axes = plt.subplots(nrows=1, ncols=len(num_cols), figsize=(15, 5))

for i, col in enumerate(num_cols):
    # gambar No dulu (besar, jadi lapisan bawah)
    sns.histplot(data=df[df['Churn']=='No'], x=col, 
                 ax=axes[i], color='steelblue', label='No')
    
    # gambar Yes di atasnya (kecil, jadi lapisan atas)
    sns.histplot(data=df[df['Churn']=='Yes'], x=col, 
                 ax=axes[i], color='orange', label='Yes')
    
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].legend()


for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
    
plt.tight_layout()
plt.show()

In [ ]:
# heatmap

df['Churn_num'] = df['Churn'].map({'Yes': 1, 'No': 0})

plt.figure(figsize=(8, 6))
sns.heatmap(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num']].corr(),
           annot=True, cmap='coolwarm',
           fmt='.2f')
plt.title('Korelasi Antar Kolom Numerik')
plt.show()


In [ ]:
df = df.drop(columns=['customerID'])

### Encode target

In [ ]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

In [ ]:
df = pd.get_dummies(df, drop_first=True)

> drop_first=True itu misalnya ada female dan male
> female 1 dan male 0 maka akan di drop 1 kolom karena dari encodenya sudah di ketahui bahwa 1 female dan 0 male

## Pemisahan data untuk training

In [ ]:
X = df.drop(columns=['Churn', 'Churn_num'])
y = df['Churn']
feature_names = X.columns.tolist()  

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Pakai model yang paling simple dulu yaitu Logistic Regresion karena ini itu klasifikasi bentuknya adalah pilihan bukan angka

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=5000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

### Penjelasan dalam analogi maling
> precision adalah misal ada maling 100 berhasil nangkap 90 jadi 90% presisi

> recal dari yang 90 maling ketanggep ternyata ada yang kabur 10 maling jadinya 90% recall

> f1 adalah nilai dari gabungan presisi dan recal

> support adalah maling itu sendiri

## Model RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(class_weight='balanced', random_state=42)
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))


## Model XGB

In [ ]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(scale_pos_weight=3, random_state=42)
model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
print(classification_report(y_test, y_pred_xgb))


## Feature Importance
> Kolom mana yang paling berpengaruh terhadap model pediksi

### Model LR

In [ ]:
coef = pd.Series(
    np.abs(model.coef_[0]),
    index=feature_names
)

coef.nlargest(10).plot(kind='barh', color='steelblue')
plt.title('Top 10 Fitur Paling Berpengaruh (Logistic Regression)')
plt.tight_layout()
plt.show()
()


### Model RF

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feat_importance = pd.Series(model_rf.feature_importances_, 
                            index=feature_names)

feat_importance.nlargest(10).plot(kind='barh')
plt.title('Top 10 Fitur Paling Berpengaruh')
plt.show()


### Model XGB

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feat_importance = pd.Series(model_xgb.feature_importances_, 
                            index=feature_names)

feat_importance.nlargest(10).plot(kind='barh')
plt.title('Top 10 Fitur Paling Berpengaruh')
plt.show()


# Kesimpulan


**Jawaban Bisnis**

1. **Berapa churn rate customer?**  
   Dari 7.032 customer setelah data cleaning, ada **1.869 customer churn** dan **5.163 tidak churn**.  
   Jadi churn rate-nya adalah **26,58%**. Artinya sekitar **1 dari 4 customer berhenti berlangganan**.

2. **Customer seperti apa yang sering churn?**  
   Customer yang paling sering churn cenderung punya karakteristik:
   - **tenure** rendah, terutama customer baru
   - kontrak **month-to-month**
   - menggunakan **Fiber optic**
   - membayar dengan **Electronic check**
   - menggunakan **Paperless Billing**
   - tidak punya layanan tambahan seperti **OnlineSecurity**, **TechSupport**, **OnlineBackup**, atau **DeviceProtection**
   - tidak punya partner/dependents

3. **Faktor apa yang paling berhubungan dengan churn?**  
   Dari korelasi numerik:
   - `tenure` punya hubungan negatif dengan churn: **-0.35**  
     Semakin lama customer berlangganan, semakin kecil kemungkinan churn.
   - `MonthlyCharges` punya hubungan positif: **0.19**  
     Semakin tinggi tagihan bulanan, churn cenderung meningkat.
   - `TotalCharges` negatif: **-0.20**, kemungkinan karena customer dengan total charge tinggi biasanya sudah lama berlangganan.

   Dari feature importance model, faktor yang sering muncul penting adalah:
   - `tenure`
   - `MonthlyCharges`
   - `TotalCharges`
   - `Contract`
   - `InternetService_Fiber optic`
   - `PaymentMethod_Electronic check`
   - `OnlineSecurity`
   - `TechSupport`

4. **Segment mana yang harus diprioritaskan untuk retention?**  
   Prioritas utama retention adalah customer dengan kombinasi risiko tinggi:

   **Customer baru + kontrak month-to-month + Fiber optic + MonthlyCharges tinggi + Electronic check + tidak punya OnlineSecurity/TechSupport.**

   Segment ini penting karena mereka punya peluang churn tinggi dan masih bisa dicegah dengan strategi retention seperti diskon awal, bundling layanan security/support, edukasi benefit paket, atau penawaran pindah ke kontrak tahunan.

Tambahan penting: karena tujuan bisnisnya adalah mendeteksi customer yang berpotensi churn, model **Logistic Regression** cukup cocok sebagai baseline karena recall churn-nya paling tinggi, yaitu **0.79**. Artinya model ini lebih baik dalam menangkap customer churn dibanding Random Forest dan XGBoost pada hasil analisis.